# H4: Foraging Profiles and Optimality (Bayesian)

All regressions use bambi Bayesian linear models with 95% HDI inference.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, os
import bambi as bmb, arviz as az
from scipy.stats import pearsonr, zscore
from scipy.special import expit
from config import *
from load_data import load_both

%matplotlib inline

exp_data, conf_data = load_both()
all_data = {'exploratory': exp_data, 'confirmatory': conf_data}
all_data = {k: v for k, v in all_data.items() if v is not None}

for name, d in all_data.items():
    print(f'{d["config"].label}: master={len(d["master"])} subjects')


Master: 290 subjects (MCMC params)


## H4a: Avoidance sensitivity predicts escape

In [ ]:
results_h4a = {}
for name, d in all_data.items():
    master = d['master']
    label = d['config'].label
    df = master[['escape_rate','omega_z','kappa_z']].dropna()
    m = bmb.Model('escape_rate ~ omega_z + kappa_z', data=df)
    r = m.fit(**BKW)
    s = az.summary(r, hdi_prob=0.95)
    lo = s.loc['omega_z','hdi_2.5%']
    passed = lo > 0
    results_h4a[name] = passed
    print(f'--- {label} ---')
    print(s[['mean','hdi_2.5%','hdi_97.5%']].to_string())
    print(f'  H4a: {"PASS" if passed else "FAIL"} (omega 95% HDI excludes zero, positive)\n')

## H4b: Overcaution + avoidance sensitivity

In [ ]:
results_h4b_pct = {}
results_h4b_reg = {}
for name, d in all_data.items():
    master = d['master']
    label = d['config'].label
    oc_pct = master['oc_ratio'].mean() * 100
    pass_pct = oc_pct > 65
    df = master[['oc_ratio','omega_z']].dropna()
    m = bmb.Model('oc_ratio ~ omega_z', data=df)
    r = m.fit(**BKW)
    s = az.summary(r, hdi_prob=0.95)
    lo = s.loc['omega_z','hdi_2.5%']
    pass_reg = lo > 0
    results_h4b_pct[name] = pass_pct
    results_h4b_reg[name] = pass_reg
    print(f'--- {label} ---')
    print(f'  Overcaution: {oc_pct:.0f}% {"PASS" if pass_pct else "FAIL"}')
    print(f'  omega -> OC: b={s.loc["omega_z","mean"]:+.4f}, 95% HDI=[{lo:+.4f},{s.loc["omega_z","hdi_97.5%"]:+.4f}]')
    print(f'  H4b: {"PASS" if pass_reg and pass_pct else "FAIL"}\n')

## H4c: Effort cost predicts pressing intensity

In [ ]:
results_h4c = {}
for name, d in all_data.items():
    master = d['master']
    label = d['config'].label
    df = master[['mean_vigor','kappa_z']].dropna()
    m = bmb.Model('mean_vigor ~ kappa_z', data=df)
    r = m.fit(**BKW)
    s = az.summary(r, hdi_prob=0.95)
    hi = s.loc['kappa_z','hdi_97.5%']
    passed = hi < 0
    results_h4c[name] = passed
    print(f'--- {label} ---')
    print(f'  kappa -> vigor: b={s.loc["kappa_z","mean"]:+.4f}, 95% HDI=[{s.loc["kappa_z","hdi_2.5%"]:+.4f},{hi:+.4f}]')
    print(f'  H4c: {"PASS" if passed else "FAIL"} (95% HDI excludes zero, negative)\n')

## H4d: Effort-driven avoidance less optimal

In [ ]:
results_h4d = {}
for name, d in all_data.items():
    master = d['master']
    label = d['config'].label
    df = master[['pct_opt','angle_z']].dropna()
    m = bmb.Model('pct_opt ~ angle_z', data=df)
    r = m.fit(**BKW)
    s = az.summary(r, hdi_prob=0.95)
    hi = s.loc['angle_z','hdi_97.5%']
    passed = hi < 0
    results_h4d[name] = passed
    print(f'--- {label} ---')
    print(f'  angle -> optimality: b={s.loc["angle_z","mean"]:+.4f}, 95% HDI=[{s.loc["angle_z","hdi_2.5%"]:+.4f},{hi:+.4f}]')
    print(f'  H4d: {"PASS" if passed else "FAIL"} (95% HDI excludes zero, negative)\n')

## H4e: W(u) consistency predicts earnings

In [ ]:
gamma = MODEL_CONSTANTS['gamma']
hazard = MODEL_CONSTANTS['hazard']
sp = MODEL_CONSTANTS['sp']
C_pen = MODEL_CONSTANTS['C_pen']
ug_np = np.linspace(0.1, 1.5, 100)

def uV(om, kap, T, D, R, req):
    speed = expit((ug_np - 0.25 * req) / sp)
    S = np.exp(-hazard * T**gamma * D / np.clip(speed, .01, None))
    W = S * R - (1 - S) * om * (R + C_pen) - kap * (ug_np - req)**2 * D
    return ug_np[W.argmax()], W.max() - kap * req * D

results_h4e_cc = {}
results_h4e_id = {}
for name, d in all_data.items():
    master = d['master']
    cdf = d['choice']
    cells = d['cell_means']
    label = d['config'].label

    ch_cons = {}; int_dev = {}
    for s in master.index:
        om = master.loc[s, 'omega']; kap = master.loc[s, 'kappa']
        st = cdf[cdf['subj'] == s]; c = 0; t = 0
        for _, tr in st.iterrows():
            _, VH = uV(om, kap, tr['threat'], tr['distance_H'], 5., .9)
            _, VL = uV(om, kap, tr['threat'], 1., 1., .4)
            if (1 if VH > VL else 0) == int(tr['choice']): c += 1
            t += 1
        ch_cons[s] = c / t if t > 0 else np.nan
        sc = cells[cells['subj'] == s]
        if len(sc) < 5: int_dev[s] = np.nan; continue
        ps = []; ac = []
        for _, cl in sc.iterrows():
            R = 5. if cl['is_heavy'] == 1 else 1.; req = .9 if cl['is_heavy'] == 1 else .4
            u, _ = uV(om, kap, cl['T_round'], cl['actual_dist'], R, req)
            ps.append(u); ac.append(cl['mean_rate'])
        int_dev[s] = np.sqrt(np.mean((np.array(ps) - np.array(ac))**2))

    cc_s = pd.Series(ch_cons).reindex(master.index)
    id_s = pd.Series(int_dev).reindex(master.index)
    valid = cc_s.notna() & id_s.notna() & master['earnings'].notna()
    master = master.copy()
    master.loc[valid, 'cc_z'] = zscore(cc_s[valid].values)
    master.loc[valid, 'id_z'] = zscore(id_s[valid].values)
    df = master[['earnings', 'cc_z', 'id_z']].dropna()
    m = bmb.Model('earnings ~ cc_z + id_z', data=df)
    r = m.fit(**BKW)
    sm = az.summary(r, hdi_prob=0.95)
    p_ch = sm.loc['cc_z', 'hdi_2.5%'] > 0
    p_vi = sm.loc['id_z', 'hdi_97.5%'] < 0
    results_h4e_cc[name] = p_ch
    results_h4e_id[name] = p_vi
    print(f'--- {label} ---')
    print(sm[['mean', 'hdi_2.5%', 'hdi_97.5%']].to_string())
    print(f'  Choice cons (b>0): {"PASS" if p_ch else "FAIL"}, Intensity dev (b<0): {"PASS" if p_vi else "FAIL"}')
    print(f'  H4e: {"PASS" if p_ch and p_vi else "FAIL"}\n')

## Summary

In [ ]:
print('H4 SUMMARY (Bayesian) — all tests use 95% HDI excludes zero')
print('=' * 70)

test_defs = [
    ('H4a', 'omega -> escape',   results_h4a),
    ('H4b', 'OC %',             results_h4b_pct),
    ('H4b', 'omega -> OC',      results_h4b_reg),
    ('H4c', 'kappa -> vigor',   results_h4c),
    ('H4d', 'angle -> opt',     results_h4d),
    ('H4e', 'choice -> earn',   results_h4e_cc),
    ('H4e', 'int_dev -> earn',  results_h4e_id),
]

sample_names = list(all_data.keys())
header = f'{"Test":<6} {"Description":<20}'
for sn in sample_names:
    header += f' {sn:<16}'
print(header)
print('-' * len(header))

for h, desc, res_dict in test_defs:
    row = f'{h:<6} {desc:<20}'
    for sn in sample_names:
        v = res_dict.get(sn, None)
        row += f' {"PASS" if v else "FAIL" if v is not None else "N/A":<16}'
    print(row)

print()
for sn in sample_names:
    n = sum(1 for _, _, rd in test_defs if rd.get(sn, False))
    total = sum(1 for _, _, rd in test_defs if sn in rd)
    print(f'{sn}: {n}/{total} tests pass')